# Anomaly Detection for P450 Plate Assay Traces

This notebook implements various anomaly detection methods for spectroscopic traces from P450 plate assays. We load data from an SQLite database, compute parameterized features, and evaluate unsupervised and supervised approaches.

## 1. Import Required Libraries and Load Data

In [4]:
%%time
import os
import json
import os.path as osp
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
from scipy import stats, optimize
import scipy
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
from sklearn.decomposition import PCA
# import torch
# import torch.nn as nn
# from torch.utils.data import Dataset, DataLoader
from imblearn.over_sampling import SMOTE

# Load data from SQLite
sql = """
select 
experiment_id,
dr.result_id,
w.id as well_id,
w.address,
r.accepted,
r.locked,
w.compound_concentration,
dr.exclude,
dr.response,
w.protein_concentration,
w.volume,
wr.well_type,
group_concat(distinct ra.comment) as comments,
group_concat(a.wavelength) as wavelength,
group_concat(a.absorbance) as absorbance
from well w
join wellresultlink wr
on wr.well_id = w.id
join result r
on r.id = wr.result_id
join doseresponse dr
on dr.result_id = wr.result_id and dr.concentration = w.compound_concentration
left join resultannotation ra
on ra.result_id = r.id
join absorbance a
on a.well_id = w.id
group by w.id
"""
with sqlite3.connect('../api/db_checkpoints/071125.db') as con:
    df = pd.read_sql(sql, con)
print(f"n results = {len(df['result_id'].unique())}")

# Process absorbance data
o = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    o.append(dict(zip(
            list(map(lambda i: int(float(i)), row['wavelength'].split(','))),
            row['absorbance'].split(','))
                 ))
o = pd.DataFrame(o).astype(float)
df = pd.concat([df, o], axis=1)
df = df.drop(['wavelength', 'absorbance'], axis=1)
print(f"Dataset shape: {df.shape}")
df.head()

n results = 914


100%|████████████████████████████████████████████████████████████████| 14624/14624 [00:03<00:00, 3766.03it/s]


Dataset shape: (14624, 594)
CPU times: user 15.7 s, sys: 1.21 s, total: 16.9 s
Wall time: 17 s


,experiment_id,result_id,well_id,address,accepted,locked,compound_concentration,exclude,response,protein_concentration,...,791,792,793,794,795,796,797,798,799,800
0,3,1,1,A1,0,1,500.00,0,1.734723e-18,8.83,...,0.027,0.028,0.028,0.029,0.029,0.028,0.029,0.028,0.025,0.025
1,3,1,2,C1,0,1,250.00,0,1.000000e-03,8.83,...,0.021,0.023,0.024,0.023,0.024,0.023,0.025,0.022,0.021,0.021
2,3,1,3,E1,0,1,125.00,0,1.734723e-18,8.83,...,0.022,0.023,0.023,0.024,0.025,0.025,0.026,0.025,0.021,0.021
3,3,1,4,G1,0,1,62.50,0,1.734723e-18,8.83,...,0.023,0.023,0.023,0.024,0.025,0.025,0.025,0.024,0.021,0.021
4,3,1,5,I1,0,1,31.25,0,1.000000e-03,8.83,...,0.022,0.022,0.022,0.023,0.024,0.024,0.025,0.023,0.020,0.018


## 2. Define Functions

In [5]:
# Utility functions
def scatter(w, k):
    return k * (1 / w**4)

def linear_fit(x):
    (m, c), cov = scipy.optimize.curve_fit(lambda x, m, c: x*m + c,
                                           xdata=range(len(x)),
                                           ydata=x
                                           )
    return m, c

# Parameterized metrics
def auc(data: pd.DataFrame, start=350, end=800):
    return np.trapz(data.loc[:, start:end], axis=1, dx=1)

def std_405(data: pd.DataFrame, wavelength=405):
    return data.loc[:, wavelength]

def dd_soret(diff_data: pd.DataFrame, start=390, end=420):
    between = diff_data.loc[:, start:end].values
    slopes = []
    for row in between:
        m, c = linear_fit(row)
        slopes.append(m)
    return np.array(slopes)

def fit_scatter(data: pd.DataFrame):
    wavelengths = data.columns.astype(float).values
    k_values = []
    for _, row in data.iterrows():
        try:
            (k,), _ = scipy.optimize.curve_fit(lambda w, k: k / w**4, wavelengths, row.values)
            k_values.append(k)
        except:
            k_values.append(np.nan)
    return np.array(k_values)

def z_score(control, test):
    mean_control = np.mean(control)
    std_control = np.std(control)
    z = (test - mean_control) / std_control
    return z

# Feature computation
def compute_features(df):
    features = pd.DataFrame(index=df.index)
    features['auc_soret'] = auc(df, start=350, end=450)
    features['auc_full'] = auc(df, start=300, end=800)
    features['val_405'] = std_405(df, wavelength=405)
    features['dd_soret'] = dd_soret(df, start=390, end=420)
    features['scatter_k'] = fit_scatter(df.loc[:, 300:800])
    return features

## 3. Feature Computation

In [6]:
# Compute features
features = compute_features(df)
features = features.fillna(0)  # Handle NaNs
y = df['exclude'].astype(int)
print(f"Features shape: {features.shape}")
features.head()

/tmp/ipykernel_601247/2218088425.py:14: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(data.loc[:, start:end], axis=1, dx=1)
/tmp/ipykernel_601247/2218088425.py:14: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(data.loc[:, start:end], axis=1, dx=1)
/tmp/ipykernel_601247/2218088425.py:6: OptimizeWarning: Covariance of the parameters could not be estimated
  (m, c), cov = scipy.optimize.curve_fit(lambda x, m, c: x*m + c,
/tmp/ipykernel_601247/2218088425.py:32: OptimizeWarning: Covariance of the parameters could not be estimated
  (k,), _ = scipy.optimize.curve_fit(lambda w, k: k / w**4, wavelengths, row.values)


Features shape: (14624, 5)


,auc_soret,auc_full,val_405,dd_soret,scatter_k
0,3.0065,19.9635,0.020,-0.000152,1.624066e+09
1,2.4680,17.1655,0.016,-0.000116,1.482639e+09
2,2.5475,17.5785,0.018,-0.000143,1.497353e+09
3,2.3990,17.1640,0.016,-0.000119,1.462255e+09
4,2.3055,16.8385,0.015,-0.000091,1.445219e+09


## 4. Z-Score Anomaly Detection

In [7]:
# Simple z-score on auc_soret
control_auc = features.loc[df['well_type'] == 'control', 'auc_soret']
z_auc = z_score(control_auc, features['auc_soret'])
anomaly_z = np.abs(z_auc) > 2.0

# Evaluate
y_pred_z = anomaly_z.astype(int)
acc = accuracy_score(y, y_pred_z)
prec = precision_score(y, y_pred_z, zero_division=0)
rec = recall_score(y, y_pred_z, zero_division=0)
f1 = f1_score(y, y_pred_z, zero_division=0)
print(f"Z-Score Detection - Acc: {acc:.2f}, Prec: {prec:.2f}, Rec: {rec:.2f}, F1: {f1:.2f}")
print(f"Baseline: {y.value_counts().max() / len(y):.2f}")

Z-Score Detection - Acc: 0.82, Prec: 0.02, Rec: 0.23, F1: 0.05
Baseline: 0.98


## 5. PCA-Based Anomaly Detection

In [8]:
# PCA on full spectra
pca = PCA(n_components=1)
pca_scores = pca.fit_transform(df.loc[:, 300:800])
z_pca = z_score(pca_scores[df['well_type'] == 'control'], pca_scores)
anomaly_pca = np.abs(z_pca) > 2.0

# Evaluate
y_pred_pca = anomaly_pca.astype(int)
acc = accuracy_score(y, y_pred_pca)
prec = precision_score(y, y_pred_pca, zero_division=0)
rec = recall_score(y, y_pred_pca, zero_division=0)
f1 = f1_score(y, y_pred_pca, zero_division=0)
print(f"PCA Detection - Acc: {acc:.2f}, Prec: {prec:.2f}, Rec: {rec:.2f}, F1: {f1:.2f}")

PCA Detection - Acc: 0.91, Prec: 0.04, Rec: 0.17, F1: 0.06


## 7. Random Forest Classification

In [9]:
# Random Forest with class_weight
X_train, X_test, y_train, y_test = train_test_split(features, y, test_size=0.2, random_state=42, stratify=y)
model_rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_estimators=100)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

acc = accuracy_score(y_test, y_pred_rf)
prec = precision_score(y_test, y_pred_rf, zero_division=0)
rec = recall_score(y_test, y_pred_rf, zero_division=0)
f1 = f1_score(y_test, y_pred_rf, zero_division=0)
try:
    auc_score = roc_auc_score(y_test, model_rf.predict_proba(X_test)[:, 1])
    print(f"RF Detection - Acc: {acc:.2f}, Prec: {prec:.2f}, Rec: {rec:.2f}, F1: {f1:.2f}, AUC: {auc_score:.2f}")
except:
    print(f"RF Detection - Acc: {acc:.2f}, Prec: {prec:.2f}, Rec: {rec:.2f}, F1: {f1:.2f}")

RF Detection - Acc: 0.98, Prec: 0.33, Rec: 0.02, F1: 0.04, AUC: 0.70


## 8. Group-Based Z-Score Detection

In [10]:
# Group-based z-score
def detect_anomalies_grouped(df, features, z_threshold=2.0):
    anomaly_flags = []
    for (result_id, well_type), group in df.groupby(['result_id', 'well_type']):
        group_features = features.loc[group.index]
        if len(group) < 2:
            anomaly_flags.extend([False] * len(group))
            continue
        z_scores = pd.DataFrame(index=group_features.index)
        for col in group_features.columns:
            mean_val = group_features[col].mean()
            std_val = group_features[col].std()
            if std_val == 0:
                z_scores[col] = 0
            else:
                z_scores[col] = (group_features[col] - mean_val) / std_val
        max_z = z_scores.abs().max(axis=1)
        flags = max_z > z_threshold
        anomaly_flags.extend(flags.values)
    return np.array(anomaly_flags)

anomaly_group = detect_anomalies_grouped(df, features, z_threshold=2.0)
y_pred_group = anomaly_group.astype(int)

acc = accuracy_score(y, y_pred_group)
prec = precision_score(y, y_pred_group, zero_division=0)
rec = recall_score(y, y_pred_group, zero_division=0)
f1 = f1_score(y, y_pred_group, zero_division=0)
print(f"Group Z-Score - Acc: {acc:.2f}, Prec: {prec:.2f}, Rec: {rec:.2f}, F1: {f1:.2f}")

Group Z-Score - Acc: 0.91, Prec: 0.05, Rec: 0.23, F1: 0.09


## 9. SMOTE Oversampling and Random Forest

In [11]:
# SMOTE + RF
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
model_smote = RandomForestClassifier(random_state=42, n_estimators=100)
model_smote.fit(X_train_sm, y_train_sm)
y_pred_smote = model_smote.predict(X_test)

acc = accuracy_score(y_test, y_pred_smote)
prec = precision_score(y_test, y_pred_smote, zero_division=0)
rec = recall_score(y_test, y_pred_smote, zero_division=0)
f1 = f1_score(y_test, y_pred_smote, zero_division=0)
try:
    auc_score = roc_auc_score(y_test, model_smote.predict_proba(X_test)[:, 1])
    print(f"SMOTE RF - Acc: {acc:.2f}, Prec: {prec:.2f}, Rec: {rec:.2f}, F1: {f1:.2f}, AUC: {auc_score:.2f}")
except:
    print(f"SMOTE RF - Acc: {acc:.2f}, Prec: {prec:.2f}, Rec: {rec:.2f}, F1: {f1:.2f}")

SMOTE RF - Acc: 0.94, Prec: 0.06, Rec: 0.15, F1: 0.09, AUC: 0.71


In [18]:
df['comments'].value_counts()[:50]

comments
No Response                                                       1056
Low Signal                                                         880
Low Response                                                       688
No Protein                                                         480
Weak Response                                                      304
Diverging Traces                                                   224
No Ligand                                                          176
Low Response,Low Signal                                            160
Type II Shift                                                      144
Low Signal,No Response                                             128
Diverging Traces,No Response                                        96
Low Response,Type II Shift                                          80
No Ligand,No Response                                               64
No Baseline                                                         

In [20]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.4/184.4 MB 4.2 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 8.1 MB/s eta 0:00:000m eta 0:00:010:00:01
  Using cached filelock-3.20.0-py3-none-any.whl (16 kB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.4/201.4 KB 3.0 MB/s eta 0:00:00 MB/s eta 0:00:01
  Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)


## 10. Neural Network Classifier with CNN for Grouped Anomalies

In [21]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [22]:
# Data preparation: Group by result_id and well_type
grouped_data = {}
for (result_id, well_type), group in df.groupby(['result_id', 'well_type']):
    spectra = group.loc[:, 300:800].values  # Spectra data
    labels = group['exclude'].values.astype(int)
    grouped_data[(result_id, well_type)] = {'spectra': spectra, 'labels': labels}

# Split groups into train/test (stratify by group)
group_keys = list(grouped_data.keys())
train_keys, test_keys = train_test_split(group_keys, test_size=0.2, random_state=42)

# Custom Dataset for variable-length groups
class GroupedSpectraDataset(Dataset):
    def __init__(self, grouped_data, keys):
        self.data = []
        for key in keys:
            spectra = torch.tensor(grouped_data[key]['spectra'], dtype=torch.float32).unsqueeze(1)  # Add channel dim
            labels = torch.tensor(grouped_data[key]['labels'], dtype=torch.float32)
            self.data.append((spectra, labels))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = GroupedSpectraDataset(grouped_data, train_keys)
test_dataset = GroupedSpectraDataset(grouped_data, test_keys)

# DataLoader with collate_fn for variable lengths
def collate_fn(batch):
    spectra = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    return spectra, labels

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

In [23]:
# Define CNN Model for variable-length sequences
class CNNAnomalyDetector(nn.Module):
    def __init__(self, input_channels=1, num_classes=1):
        super(CNNAnomalyDetector, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 32, kernel_size=5, stride=1, padding=2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, stride=1, padding=2)
        self.pool = nn.AdaptiveAvgPool1d(1)  # Global average pooling for variable lengths
        self.fc = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x.squeeze(-1).unsqueeze(-1)))  # Adjust for pooling
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = CNNAnomalyDetector()

In [24]:
# Handle class imbalance: Compute class weights
all_labels = []
for key in train_keys:
    all_labels.extend(grouped_data[key]['labels'])
pos_weight = len([l for l in all_labels if l == 0]) / len([l for l in all_labels if l == 1])
pos_weight = torch.tensor(pos_weight, dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [25]:
# Training loop
num_epochs = 10
model.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    for spectra_batch, labels_batch in train_loader:
        for spectra, labels in zip(spectra_batch, labels_batch):
            optimizer.zero_grad()
            outputs = model(spectra)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
    logger.info(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss / len(train_loader):.4f}")

2025-12-13 18:16:27,814 - INFO - Epoch 1/10, Loss: 1.3784
2025-12-13 18:16:30,372 - INFO - Epoch 2/10, Loss: 1.3627
2025-12-13 18:16:33,126 - INFO - Epoch 3/10, Loss: 1.3610
2025-12-13 18:16:36,351 - INFO - Epoch 4/10, Loss: 1.3584
2025-12-13 18:16:38,877 - INFO - Epoch 5/10, Loss: 1.3510
2025-12-13 18:16:41,433 - INFO - Epoch 6/10, Loss: 1.3474
2025-12-13 18:16:44,130 - INFO - Epoch 7/10, Loss: 1.3440
2025-12-13 18:16:46,801 - INFO - Epoch 8/10, Loss: 1.3373
2025-12-13 18:16:49,539 - INFO - Epoch 9/10, Loss: 1.3489
2025-12-13 18:16:52,226 - INFO - Epoch 10/10, Loss: 1.3472


In [26]:
# Evaluation
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for spectra_batch, labels_batch in test_loader:
        for spectra, labels in zip(spectra_batch, labels_batch):
            outputs = model(spectra)
            preds = torch.sigmoid(outputs).squeeze() > 0.5
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)
try:
    auc_score = roc_auc_score(all_labels, all_preds.astype(int))
    print(f"CNN Detection - Acc: {acc:.2f}, Prec: {prec:.2f}, Rec: {rec:.2f}, F1: {f1:.2f}, AUC: {auc_score:.2f}")
except:
    print(f"CNN Detection - Acc: {acc:.2f}, Prec: {prec:.2f}, Rec: {rec:.2f}, F1: {f1:.2f}")

CNN Detection - Acc: 0.65, Prec: 0.01, Rec: 0.28, F1: 0.03


In [28]:
df[800]

0        0.025
1        0.021
2        0.021
3        0.021
4        0.018
         ...  
14619    0.090
14620    0.061
14621    0.042
14622    0.042
14623    0.035
Name: 800, Length: 14624, dtype: float64